# Keyword normalization and service sub-aspects

Input: the keyword files exported by `queries/xpath_queries.sh` (one file per brand, aspect and rating).

For every SERVICE keyword phrase: tokenize, remove stop words, lemmatize with POS tags,
assign a service sub-aspect with a rule-based lexicon, and score subjectivity with TextBlob.

Output: `keywords_dict_expanded.parquet`, one row per normalized keyword, used by `scripts/import_to_postgres.py`
and `notebooks/build_knowledge_graph.ipynb`.

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from textblob import TextBlob
import os
import pandas as pd
import numpy as np

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

### Set the POS categories for lemmatization 

In [ ]:
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        # As a default, assume it's a noun
        return wordnet.NOUN
    

### Processing

In [ ]:
# --- Tokenization of the keywords
def tokenization(phrase):
    tokens = word_tokenize(phrase.lower())
    stop_words = set(stopwords.words('english'))
    filtered_words = [word for word in tokens if word.isalnum() and word not in stop_words]
    
    return filtered_words

# --- Lemmatization of the keywords
def lemmatization(filtered_words):
    if type(filtered_words) != list:
        filtered_words = [filtered_words]
    tagged_tokens = nltk.pos_tag(filtered_words)
    lemmatizer = WordNetLemmatizer()
    lemmatized_words = []
    for word, tag in tagged_tokens:
        pos = get_wordnet_pos(tag)
        lemmatized_words.append(lemmatizer.lemmatize(word, pos=pos))
    
    return lemmatized_words

# --- rule-based keywords categorization
def get_aspect(words):
    lexicon = {
    "TIMELINESS": ["minute", "wait", "waited", "slow", "fast", "quick", "hour", "forever", "efficient", "urgency", "speed", "stuck", "long",
                    "waste","efficiently", "quickly"],
    "STAFF_ATTITUDE": ["rude", "attitude", "horrid", "suck", "nice", "friendly", "polite", "non-enthusiastic", "impatient", "unfriendly",
                       "humor", "gracious", "sweet", "nice", "bright", "stare", "blank", "demean", "awkward", "dereliction", "lazy",
                       "ignorant", "inattentive", "jackass", "horrible", "asshole", "disrespectful", "apologetic", "rudely", "attention",
                       "finger", "humiliate"],
    "ORDER_ACCURACY": ["wrong", "forget", "miss", "correct", "screw"],
    "STAFFING_LEVELS": ["understaffed", "undertrained", "staff", "inability", "professional"],
    }
    for aspect, keywords in lexicon.items():
        if any(word in keywords for word in words):
            return aspect

    print(f"{words} not categorized.")
    return "OTHER"
        
# --- get the subjectivity of keywords using textblob
def get_subjectivity(words):
    blob_words = TextBlob(words)
    subjectivity = blob_words.sentiment.subjectivity
    return subjectivity
    

### Parsing

In [ ]:
def parsing(file):
    word_dict = []
    aspects = []
    subjectivity = []
    for line in file:
        tokenized_words = tokenization(line)
        filtered_words = lemmatization(tokenized_words)
        word_dict.append(filtered_words)
        aspect = get_aspect(filtered_words)
        aspects.append(aspect)
        subjectivity.append(get_subjectivity(line))
    return word_dict, aspects, subjectivity
    

### Main loop

In [ ]:
file_folder = "keywords"  # output folder of queries/xpath_queries.sh
df_all = pd.DataFrame()
for filename in os.listdir(file_folder):
    print(filename)
    if filename.find('service') != -1:
        tokens = filename.split('_')
        brand = tokens[0]
        sentiment = tokens[-1].replace('.txt', '')
        print(sentiment)
        file_path = os.path.join(file_folder, f"{filename}")
        print(file_path)
        with open(file_path, 'r') as f:
            print(file_path)
            word_dict, aspects, subjectivity = parsing(f)
        data = {'BRAND':brand,
                'SENTIMENT':sentiment,
                'KEYWORDS':word_dict,
                'ASPECT':aspects,
                'SUBJECTIVITY':subjectivity}
        df = pd.DataFrame(data)
        df_all = pd.concat([df_all, df])

print(df_all.info())
print(df_all.head())
print(df_all.tail())
            

In [ ]:
output_path = "keywords_dict.pkl"
df_all.to_pickle(output_path)

In [ ]:
df_all = df_all.dropna(subset='KEYWORDS')
df_all['REVIEW'] = df_all['KEYWORDS']
df_all['REVIEW_NUM'] = np.arange(1, len(df_all) + 1)
df_expanded = df_all.explode('KEYWORDS')
print(df_expanded.info())
print(df_expanded.head())
print(df_expanded.tail())
df_expanded.to_pickle("keywords_dict_expanded.pkl")

In [ ]:
df_expanded = df_all.explode('KEYWORDS')
df_expanded.to_parquet("keywords_dict_expanded.parquet")

In [ ]:
print(df_expanded.tail())
print(df_expanded.info())